# Tracking Azure Machine Learning Jobs with MLflow


# Notebook Setup

Set project paths and load workspace MLClient.

In [4]:
import os
# Here we set the working directory to the project root to ensure imports work correctly
from pathlib import Path
target = "dp100-learn"
p = Path.cwd()
print(f"Starting working directory: {p}")
while p.name != target and p.parent != p:
    p = p.parent
# Set the path to your project root manually if the above code does not work
p = "/mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn"
# p = "C:/Users/dmika/DEV/Projects-local/dp100-learn"
os.chdir(p)
print("Changed working directory to:", p)
from utils.azureml_utils import *

# Get Azure ML Client based on your environment. Learn more in the tutorials/azureml-first-notebook.ipynb.
ml_client = get_azureml_client()

Starting working directory: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code
Changed working directory to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn
Added to sys.path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn


Found the config file in: /config.json


Added to sys.path: /mnt/batch/tasks/shared/LS_root/mounts/clusters/ci-dm-dp100-cpu123814/code/Users/dominik.mika/dp100-learn


In [5]:
import mlflow

# Track model training with MLflow

## Create an MLflow experiment

You'll first create an MLflow experiment. By creating the experiment, you can group all runs within one experiment and make it easier to find the runs in the studio.

In [13]:
experiment_name = "dmdp100-mlflow-basic-tracking"
mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='', creation_time=1763484884452, experiment_id='2b2fb059-45ac-45ac-9cc0-6b32d6e81d15', last_update_time=None, lifecycle_stage='active', name='dmdp100-mlflow-basic-tracking', tags={}>

## Prepare the data

You'll train a diabetes classification model. The training data is stored in the **data** folder as **diabetes.csv**. 

First, let's read the data:

In [10]:
import pandas as pd

print("Reading data...")

diabetes_df_asset = ml_client.data.get(name="diabetes-data-file", version="1")
df = pd.read_csv(diabetes_df_asset.path)
df.head()

Reading data...


Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,PatientID,Pregnancies,PlasmaGlucose,DiastolicBloodPressure,TricepsThickness,SerumInsulin,BMI,DiabetesPedigree,Age,Diabetic
0,1354778,0,171,80,34,23,43.509726,1.213191,21,0
1,1147438,8,92,93,47,36,21.240576,0.158365,23,0
2,1640031,7,115,47,52,35,41.511523,0.079019,23,0
3,1883350,9,103,78,25,304,29.582192,1.282870,43,1
4,1424119,1,85,59,27,35,42.604536,0.549542,22,0


Next, you'll split the data into features and the label (Diabetes):

In [11]:
print("Splitting data...")
X, y = df[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness','SerumInsulin','BMI','DiabetesPedigree','Age']].values, df['Diabetic'].values

Splitting data...


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

You now have four dataframes:

- `X_train`: The training dataset containing the features.
- `X_test`: The test dataset containing the features.
- `y_train`: The label for the training dataset.
- `y_test`: The label for the test dataset.

You'll use these to train and evaluate the models you'll train.

## Train and track models

To track a model you train, you can use MLflow and enable autologging. The following cell will train a classification model using logistic regression. You'll notice that you don't need to calculate any evaluation metrics because they're automatically created and logged by MLflow.

In [14]:
from sklearn.linear_model import LogisticRegression

with mlflow.start_run(run_name="logistic-regression-diabetes-autolog-1"):
    mlflow.sklearn.autolog()
    model = LogisticRegression(C=1/0.1, solver="liblinear").fit(X_train, y_train)


🏃 View run logistic-regression-diabetes-autolog-1 at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15/runs/deb4bf8d-a880-4366-adc2-907b3781b9c2
🧪 View experiment at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15


You can also use custom logging with MLflow. You can add custom logging to autologging, or you can use only custom logging.

Let's train two more models with scikit-learn. Since you ran the `mlflow.sklearn.autolog()` command before, MLflow will now automatically log any model trained with scikit-learn. To disable the autologging, run the following cell:

In [15]:
mlflow.sklearn.autolog(disable=True)

Now, you can train and track models using only custom logging. 

When you run the following cell, you'll only log one parameter and one metric.

In [16]:
from sklearn.linear_model import LogisticRegression
import numpy as np

with mlflow.start_run(run_name="logistic-regression-diabetes-custom-logging-1"):
    model = LogisticRegression(C=1/0.1, solver="liblinear").fit(X_train, y_train)

    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)

    mlflow.log_param("regularization_rate", 0.1)
    mlflow.log_metric("Accuracy", acc)

🏃 View run logistic-regression-diabetes-custom-logging-1 at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15/runs/ca7965e7-10a3-422e-b516-0174e4f28d48
🧪 View experiment at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15


The reason why you'd want to track models, could be to compare the results of models you train with different hyperparameter values. 

For example, you just trained a logistic regression model with a regularization rate of 0.1. Now, train another model, but this time with a regularization rate of 0.01. Since you're also tracking the accuracy, you can compare and decide which rate results in a better performing model.

In [17]:
from sklearn.linear_model import LogisticRegression
import numpy as np

with mlflow.start_run(run_name="logistic-regression-diabetes-custom-logging-2"):
    model = LogisticRegression(C=1/0.01, solver="liblinear").fit(X_train, y_train)

    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)

    mlflow.log_param("regularization_rate", 0.01)
    mlflow.log_metric("Accuracy", acc)

🏃 View run logistic-regression-diabetes-custom-logging-2 at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15/runs/8f3bcb7e-1da4-44a9-b892-b457ef2686c8
🧪 View experiment at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15


Another reason to track your model's results is when you're testing another estimator. All models you've trained so far used the logistic regression estimator. 

Run the following cell to train a model with the decision tree classifier estimator and review whether the accuracy is higher compared to the other runs.

In [18]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np

with mlflow.start_run(run_name="decision-tree-diabetes-custom-logging-1"):
    model = DecisionTreeClassifier().fit(X_train, y_train)

    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)

    mlflow.log_param("estimator", "DecisionTreeClassifier")
    mlflow.log_metric("Accuracy", acc)

🏃 View run decision-tree-diabetes-custom-logging-1 at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15/runs/be5582d3-263c-4152-a3e3-35f2313da1b4
🧪 View experiment at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/2b2fb059-45ac-45ac-9cc0-6b32d6e81d15


Finally, let's try to log an artifact. An artifact can be any file. For example, you can plot the ROC curve and store the plot as an image. The image can be logged as an artifact. 

Run the following cell to log a parameter, metric, and an artifact.

In [24]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt
import numpy as np

with mlflow.start_run(run_name="decision-tree-diabetes-custom-logging-2"):
    model = DecisionTreeClassifier().fit(X_train, y_train)

    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)

    # plot ROC curve
    y_scores = model.predict_proba(X_test)

    fpr, tpr, thresholds = roc_curve(y_test, y_scores[:,1])
    fig = plt.figure(figsize=(6, 4))
    # Plot the diagonal 50% line
    plt.plot([0, 1], [0, 1], 'k--')
    # Plot the FPR and TPR achieved by our model
    plt.plot(fpr, tpr)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.savefig("artifacts/figs/ROC-Curve.png")

    mlflow.log_param("estimator", "DecisionTreeClassifier")
    mlflow.log_metric("Accuracy", acc)
    mlflow.log_artifact("artifacts/figs/ROC-Curve.png")

🏃 View run decision-tree-diabetes-custom-logging-2 at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/fbd0f4ed-abbc-47e5-b20b-5fb825334ba2/runs/35a2472c-2b9c-459f-bd0a-8576895ae059
🧪 View experiment at: https://westeurope.api.azureml.ms/mlflow/v2.0/subscriptions/d34fa9f4-747f-4537-a7bf-b67844e5e3c3/resourceGroups/polandaidevml-rg/providers/Microsoft.MachineLearningServices/workspaces/polandaidevml-mlw/#/experiments/fbd0f4ed-abbc-47e5-b20b-5fb825334ba2


Review the model's results on the Jobs page of the Azure Machine Learning studio. 

- You'll find the parameters under **Params** in the **Overview** tab.
- You'll find the metrics under **Metrics** in the **Overview** tab, and in the **Metrics** tab.
- You'll find the artifacts in the **Outputs + logs** tab.


# Use MLflow to track Jobs

In [21]:
experiment_name = "dmdp100-mlflow-job-tracking"
mlflow.set_experiment(experiment_name)

2025/11/18 17:05:57 INFO mlflow.tracking.fluent: Experiment with name 'dmdp100-mlflow-job-tracking' does not exist. Creating a new experiment.


<Experiment: artifact_location='', creation_time=1763485557559, experiment_id='fbd0f4ed-abbc-47e5-b20b-5fb825334ba2', last_update_time=None, lifecycle_stage='active', name='dmdp100-mlflow-job-tracking', tags={}>

In [27]:
tutorials_materials_src_path = "utils/tutorials-materials/src"


## Custom tracking with MLflow

When running a script as a job you can use MLflow in your training script to track the model. MLflow allows you to track any custom parameters, metrics, or artifacts you want to store with your job output.

Run the following cells to create the **train-model-mlflow.py** script in the **src** folder. The script trains a classification model by using the **diabetes.csv** file in the same folder, which is passed as an argument. 

Review the code below to find that the script will import `mlflow` and log:

- The regularization rate as a **parameter**. 
- The accuracy and AUC as **metrics**.
- The plotted ROC curve as an **artifact**.

In [26]:
%%writefile $tutorials_materials_src_path/train-model-mlflow.py
# import libraries
import mlflow
import argparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

def main(args):
    # read data
    df = get_data(args.training_data)

    # split data
    X_train, X_test, y_train, y_test = split_data(df)

    # train model
    model = train_model(args.reg_rate, X_train, X_test, y_train, y_test)

    # evaluate model
    eval_model(model, X_test, y_test)

# function that reads the data
def get_data(path):
    print("Reading data...")
    df = pd.read_csv(path)
    
    return df

# function that splits the data
def split_data(df):
    print("Splitting data...")
    X, y = df[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness',
    'SerumInsulin','BMI','DiabetesPedigree','Age']].values, df['Diabetic'].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

    return X_train, X_test, y_train, y_test

# function that trains the model
def train_model(reg_rate, X_train, X_test, y_train, y_test):
    mlflow.log_param("Regularization rate", reg_rate)
    print("Training model...")
    model = LogisticRegression(C=1/reg_rate, solver="liblinear").fit(X_train, y_train)

    return model

# function that evaluates the model
def eval_model(model, X_test, y_test):
    # calculate accuracy
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)
    print('Accuracy:', acc)
    mlflow.log_metric("Accuracy", acc)

    # calculate AUC
    y_scores = model.predict_proba(X_test)
    auc = roc_auc_score(y_test,y_scores[:,1])
    print('AUC: ' + str(auc))
    mlflow.log_metric("AUC", auc)

    # plot ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_scores[:,1])
    fig = plt.figure(figsize=(6, 4))
    # Plot the diagonal 50% line
    plt.plot([0, 1], [0, 1], 'k--')
    # Plot the FPR and TPR achieved by our model
    plt.plot(fpr, tpr)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.savefig("ROC-Curve.png")
    mlflow.log_artifact("ROC-Curve.png")    

def parse_args():
    # setup arg parser
    parser = argparse.ArgumentParser()

    # add arguments
    parser.add_argument("--training_data", dest='training_data',
                        type=str)
    parser.add_argument("--reg_rate", dest='reg_rate',
                        type=float, default=0.01)

    # parse args
    args = parser.parse_args()

    # return args
    return args

# run script
if __name__ == "__main__":
    # add space in logs
    print("\n\n")
    print("*" * 60)

    # parse args
    args = parse_args()

    # run main function
    main(args)

    # add space in logs
    print("*" * 60)
    print("\n\n")


Writing utils/tutorials-materials/src/train-model-mlflow.py


# Other

Uncategorized code snippets

In [4]:
import mlflow

mlflow.set_experiment(experiment_name="dmdp100-tracking-playground")

2025/11/18 16:45:43 INFO mlflow.tracking.fluent: Experiment with name 'dmdp100-tracking-playground' does not exist. Creating a new experiment.


<Experiment: artifact_location='', creation_time=1763484342993, experiment_id='c6a82534-f9d7-4ea1-8b3a-6e23776c04a4', last_update_time=None, lifecycle_stage='active', name='dmdp100-tracking-playground', tags={}>

In [3]:
!pip show azureml-mlflow

Name: azureml-mlflow
Version: 1.60.0
Summary: Contains the integration code of AzureML with Mlflow.
Home-page: https://docs.microsoft.com/python/api/overview/azure/ml/?view=azure-ml-py
Author: Microsoft Corp
Author-email: 
License: https://aka.ms/azureml-sdk-license
Location: /anaconda/envs/azureml_py38/lib/python3.10/site-packages
Requires: azure-common, azure-core, azure-identity, azure-mgmt-core, azure-storage-blob, cryptography, jsonpickle, mlflow-skinny, msrest, python-dateutil, pytz
Required-by: azureml-train-automl-runtime
